# 02 — Network Probe

This notebook:
- Loads candidate URLs from the asset catalog
- Probes each with HTTP HEAD/GET
- Attempts OGC GetCapabilities for promising endpoints
- Produces a probe results CSV

In [ ]:
from __future__ import annotations

import pathlib
import pandas as pd
import httpx
from gdynia_thermal_audit.settings import Settings
from gdynia_thermal_audit.logging_utils import setup_logging
from gdynia_thermal_audit.network_probe.endpoint_probe import probe_endpoint
from gdynia_thermal_audit.network_probe.service_discovery import discover_services
from gdynia_thermal_audit.network_probe.capabilities import fetch_wms_capabilities, fetch_wmts_capabilities, fetch_wfs_capabilities
from gdynia_thermal_audit.utils.io import ensure_dir

setup_logging()
settings = Settings()

In [ ]:
INTERIM_DIR = pathlib.Path(settings.data_dir) / 'interim'
ensure_dir(INTERIM_DIR)

# Load candidate URLs from asset catalog (or demo)
catalog_path = INTERIM_DIR / 'asset_catalog.csv'
if catalog_path.exists():
    df_catalog = pd.read_csv(catalog_path)
else:
    # Fall back to demo data
    df_catalog = pd.read_csv('data/demo/demo_source_inventory.csv')
    print('Using demo data')

candidate_urls = df_catalog['url'].dropna().tolist()
print(f'{len(candidate_urls)} candidate URLs loaded')

In [ ]:
# Probe all candidate endpoints
results = []
with httpx.Client(timeout=settings.request_delay_s * 10, follow_redirects=True,
                  headers={'User-Agent': settings.user_agent}) as session:
    for url in candidate_urls[:20]:  # limit for notebook run
        result = probe_endpoint(url, session=session, delay=settings.request_delay_s)
        results.append(result.__dict__ if hasattr(result, '__dict__') else dict(result))

df_probe = pd.DataFrame(results)
df_probe.to_csv(INTERIM_DIR / 'probe_results.csv', index=False)
df_probe.head()

In [ ]:
# Check OGC capabilities for HTTP 200 endpoints with XML content-type
wms_candidates = df_probe[
    df_probe['status_code'].eq(200) &
    df_probe['content_type'].fillna('').str.contains('xml', case=False)
]['url'].tolist()

print(f'{len(wms_candidates)} potential OGC endpoints found')
for url in wms_candidates:
    print(f'  Checking: {url}')